# Create second-stage train, validation, and test splits

This notebook reads `Data/Raw/Descriptor_ss.csv`, containing grain-size classes 11, 12, and 13, then writes `ss_`-prefixed split metadata into `Data/Preprocessed`.

It does not copy or move matrix `.txt` files. It validates annotated target-cell files by direct path lookup under `Data/Raw/beach_all/<date>/`; later notebooks use the saved `date` and `cell_id` values to load target cells and their same-date context from that directory.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor_ss.csv"
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

# Fractions are applied to descriptor rows. Keep their sum equal to 1.0.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RANDOM_SEED = 42

assert abs((TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION) - 1.0) < 1e-9

print(f"Project root:     {PROJECT_ROOT}")
print(f"Descriptor:       {DESCRIPTOR_PATH}")
print(f"Cell data root:   {CELL_DATA_ROOT}")
print(f"Output directory: {PREPROCESSED_DIR}")

Project root:     /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach
Descriptor:       /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/Descriptor_ss.csv
Cell data root:   /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/beach_all
Output directory: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed


In [3]:
SOURCE_COLUMNS = {
    "Site": "site",
    "Date": "date",
    "Cell": "cell_id",
    "d50": "d50",
    "Class": "class",
}
DESCRIPTOR_COLUMNS = ["site", "date", "cell_id", "d50", "class"]
DESCRIPTOR_DTYPES = {column: "string" for column in DESCRIPTOR_COLUMNS}


def read_descriptor(path):
    if not path.exists():
        raise FileNotFoundError(f"Descriptor file not found: {path}")
    frame = pd.read_csv(path, dtype="string", encoding="utf-8-sig")
    missing_columns = set(SOURCE_COLUMNS) - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Descriptor is missing columns: {sorted(missing_columns)}")
    frame = frame.rename(columns=SOURCE_COLUMNS)[DESCRIPTOR_COLUMNS]

    frame = frame.dropna(subset=DESCRIPTOR_COLUMNS).copy()
    for column in DESCRIPTOR_COLUMNS:
        frame[column] = frame[column].str.strip()
    empty_mask = frame[DESCRIPTOR_COLUMNS].eq("").any(axis=1)
    if empty_mask.any():
        raise ValueError(f"Empty descriptor values found in {path}: {int(empty_mask.sum())} rows")
    return frame


descriptor_raw = read_descriptor(DESCRIPTOR_PATH)
expected_classes = {"11", "12", "13"}
actual_classes = set(descriptor_raw["class"])
if actual_classes != expected_classes:
    raise ValueError(
        f"Expected second-stage classes {sorted(expected_classes)}, got {sorted(actual_classes)}"
    )

duplicate_mask = descriptor_raw.duplicated(subset=["date", "cell_id"], keep=False)
if duplicate_mask.any():
    duplicates = descriptor_raw.loc[duplicate_mask].sort_values(["date", "cell_id", "class"])
    conflicting_keys = (
        duplicates.groupby(["date", "cell_id"])["class"].nunique().loc[lambda counts: counts > 1]
    )
    if not conflicting_keys.empty:
        conflicts = duplicates.merge(
            conflicting_keys.rename("class_count"),
            on=["date", "cell_id"],
        )
        display(conflicts)
        raise ValueError("Conflicting classes found for duplicate date/cell_id pairs.")
    duplicate_key_count = duplicates[["date", "cell_id"]].drop_duplicates().shape[0]
    print(f"Dropping {len(duplicates) - duplicate_key_count:,} repeated annotations.")

descriptor = (
    descriptor_raw
    .drop_duplicates(subset=["date", "cell_id"], keep="first")
    .sort_values(["date", "cell_id"])
    .reset_index(drop=True)
)
descriptor.head(), descriptor.shape

(  site      date  cell_id   d50 class
 0   67  20220826  1000972  40.1    13
 1   68  20220826  1031549  20.9    12
 2   69  20220826  1045306  17.5    12
 3   70  20220826  1069762  23.7    12
 4   71  20220826  1094222  34.5    13,
 (140, 5))

In [4]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Cell data root not found: {CELL_DATA_ROOT}")

missing_rows = []
for row in descriptor.to_dict(orient="records"):
    missing_features = [
        suffix
        for suffix in FEATURE_SUFFIXES
        if not (CELL_DATA_ROOT / row["date"] / f"{row['cell_id']}{suffix}.txt").is_file()
    ]
    if missing_features:
        missing_rows.append(
            {
                "date": row["date"],
                "cell_id": row["cell_id"],
                "class": row["class"],
                "missing_features": ",".join(missing_features),
            }
        )

missing_cells = pd.DataFrame(missing_rows)
if not missing_cells.empty:
    display(missing_cells.head(50))
    raise FileNotFoundError(
        f"{len(missing_cells):,} annotated cells are incomplete under {CELL_DATA_ROOT}."
    )

print(f"Validated five feature files for all {len(descriptor):,} annotated cells in {CELL_DATA_ROOT}.")

class_counts = descriptor["class"].value_counts().sort_index()
class_counts.to_frame(name="count")

Validated five feature files for all 140 annotated cells in /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Raw/beach_all.


,count
class,
11,22
12,64
13,54


In [5]:
train_df, holdout_df = train_test_split(
    descriptor,
    train_size=TRAIN_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=descriptor["class"],
)

validation_relative_fraction = VALIDATION_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION)
validation_df, test_df = train_test_split(
    holdout_df,
    train_size=validation_relative_fraction,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=holdout_df["class"],
)

train_df = train_df.assign(split="train")
validation_df = validation_df.assign(split="validation")
test_df = test_df.assign(split="test")

split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)
split_df = split_df.sort_values(["split", "date", "cell_id"]).reset_index(drop=True)

split_counts = split_df["split"].value_counts().reindex(["train", "validation", "test"])
split_counts.to_frame(name="count")

,count
split,
train,98
validation,21
test,21


In [6]:
expected_keys = set(map(tuple, descriptor[["date", "cell_id"]].to_numpy()))
actual_keys = set(map(tuple, split_df[["date", "cell_id"]].to_numpy()))

if expected_keys != actual_keys:
    missing = expected_keys - actual_keys
    unexpected = actual_keys - expected_keys
    raise AssertionError(f"Split mismatch. Missing={len(missing)}, unexpected={len(unexpected)}")

if split_df.duplicated(subset=["date", "cell_id"]).any():
    raise AssertionError("A date/cell_id pair appears in more than one split.")

distribution = (
    split_df
    .groupby(["split", "class"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "validation", "test"])
)

distribution

class,11,12,13
split,,,
train,15,45,38
validation,3,10,8
test,4,9,8


In [7]:
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

columns = ["site", "date", "cell_id", "d50", "class", "split"]
output_files = {
    "all": PREPROCESSED_DIR / "gs_dataset_splits.csv",
    "train": PREPROCESSED_DIR / "gs_train_split.csv",
    "validation": PREPROCESSED_DIR / "gs_validation_split.csv",
    "test": PREPROCESSED_DIR / "gs_test_split.csv",
}

split_df[columns].to_csv(output_files["all"], index=False)
for split_name in ["train", "validation", "test"]:
    split_df.loc[split_df["split"] == split_name, columns].to_csv(output_files[split_name], index=False)

for split_name, path in output_files.items():
    print(f"{split_name:>10}: {path}")

       all: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_dataset_splits.csv
     train: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_train_split.csv
validation: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_validation_split.csv
      test: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/pmaciag-volume-beach/Data/Preprocessed/gs_test_split.csv
